# Module 01 — Spark API & Query Plans

Goals: PySpark DataFrame API vs `spark.sql`, one engine underneath; read plans from logical → physical.

Open the **Application UI** (host port mapped to container `4040`) before running actions.

In [1]:
import sys
sys.path.insert(0, '/workspace')
from shared.spark_session import begin_module, ui_links, explain_plan
from shared import config
from shared.utils import timed_action, timed_read, drop_iceberg_table, reset_spark_configs
from shared.demo_data import user_post_activity

spark = begin_module('module-01-spark-plans')
ui_links()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/17 08:01:46 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


{'master_ui': 'http://localhost:8080',
 'app_ui': 'http://localhost:4040',
 'worker_1_ui': 'http://localhost:8081',
 'worker_2_ui': 'http://localhost:8082',
 'app_ui_internal': 'http://jupyter-lab:4040'}

## TLC data

In [9]:
trips = spark.read.parquet(*config.yellow_taxi_paths())
trips.createOrReplaceTempView("trips")
zones = spark.read.option('header', True).csv(config.zones_path())
print('trips columns:', len(trips.columns))

trips columns: 19


## PySpark vs Spark SQL — same query

Spark parses both into a **logical plan**, runs **analyzer** (resolve columns/types), **optimizer** (rules), then builds a **physical plan** for execution.

In [10]:
from pyspark.sql import functions as F

df_api = (
    trips.filter(F.col('VendorID').isin(1, 2))
    .groupBy('VendorID')
    .agg(F.count('*').alias('trips'), F.avg('total_amount').alias('avg_fare'))
    .orderBy(F.desc('trips'))
)
df_api.show(5)

+--------+-------+------------------+
|VendorID|  trips|          avg_fare|
+--------+-------+------------------+
|       2|4506016| 26.99157657009724|
|       1|1465874|25.850190834957132|
+--------+-------+------------------+



In [12]:
df_sql = spark.sql('''
    SELECT VendorID, COUNT(*) AS trips, AVG(total_amount) AS avg_fare
    FROM trips
    WHERE VendorID IN (1, 2)
    GROUP BY VendorID
    ORDER BY trips DESC
'''.format(config.yellow_taxi_paths()[0]))
df_sql.show(5)

+--------+-------+-----------------+
|VendorID|  trips|         avg_fare|
+--------+-------+-----------------+
|       2|4506016|26.99157657009724|
|       1|1465874|25.85019083495713|
+--------+-------+-----------------+



In [13]:
print('--- DataFrame API plan ---')
df_api.explain('formatted')
print('--- Spark SQL plan ---')
df_sql.explain('formatted')

--- DataFrame API plan ---
== Physical Plan ==
AdaptiveSparkPlan (8)
+- Sort (7)
   +- Exchange (6)
      +- HashAggregate (5)
         +- Exchange (4)
            +- HashAggregate (3)
               +- Filter (2)
                  +- Scan parquet  (1)


(1) Scan parquet 
Output [2]: [VendorID#256, total_amount#272]
Batched: true
Location: InMemoryFileIndex [file:/workspace/data/cache/yellow_tripdata_2024-01.parquet, ... 1 entries]
PushedFilters: [In(VendorID, [1,2])]
ReadSchema: struct<VendorID:int,total_amount:double>

(2) Filter
Input [2]: [VendorID#256, total_amount#272]
Condition : VendorID#256 IN (1,2)

(3) HashAggregate
Input [2]: [VendorID#256, total_amount#272]
Keys [1]: [VendorID#256]
Functions [2]: [partial_count(1), partial_avg(total_amount#272)]
Aggregate Attributes [3]: [count#323L, sum#324, count#325L]
Results [4]: [VendorID#256, count#326L, sum#327, count#328L]

(4) Exchange
Input [4]: [VendorID#256, count#326L, sum#327, count#328L]
Arguments: hashpartitioning(VendorID#

## Extended `explain(true)` — plan phases

In [14]:
df_api.explain(mode='extended')

== Parsed Logical Plan ==
'Sort ['trips DESC NULLS LAST], true
+- Aggregate [VendorID#256], [VendorID#256, count(1) AS trips#297L, avg(total_amount#272) AS avg_fare#298]
   +- Filter VendorID#256 IN (1,2)
      +- Relation [VendorID#256,tpep_pickup_datetime#257,tpep_dropoff_datetime#258,passenger_count#259L,trip_distance#260,RatecodeID#261L,store_and_fwd_flag#262,PULocationID#263,DOLocationID#264,payment_type#265L,fare_amount#266,extra#267,mta_tax#268,tip_amount#269,tolls_amount#270,improvement_surcharge#271,total_amount#272,congestion_surcharge#273,Airport_fee#274] parquet

== Analyzed Logical Plan ==
VendorID: int, trips: bigint, avg_fare: double
Sort [trips#297L DESC NULLS LAST], true
+- Aggregate [VendorID#256], [VendorID#256, count(1) AS trips#297L, avg(total_amount#272) AS avg_fare#298]
   +- Filter VendorID#256 IN (1,2)
      +- Relation [VendorID#256,tpep_pickup_datetime#257,tpep_dropoff_datetime#258,passenger_count#259L,trip_distance#260,RatecodeID#261L,store_and_fwd_flag#262,

### Vocabulary

| Phase | Role |
|-------|------|
| **Parsed / Logical** | What you asked for |
| **Analyzed** | Catalog resolved, types fixed |
| **Optimized** | Filter pushdown, column pruning (more in module 02) |
| **Physical** | Joins, shuffle, scans chosen for executors |

## Example 2 — projection + filter (DataFrame & SQL)

In [15]:
narrow_api = trips.select('VendorID', 'total_amount', 'PULocationID').filter('total_amount > 50')
narrow_sql = spark.sql(f'''
    SELECT VendorID, total_amount, PULocationID
    FROM trips
    WHERE total_amount > 50
''')
explain_plan(narrow_api)
narrow_sql.explain('formatted')

== Physical Plan ==
* Project (4)
+- * Filter (3)
   +- * ColumnarToRow (2)
      +- Scan parquet  (1)


(1) Scan parquet 
Output [3]: [VendorID#256, PULocationID#263, total_amount#272]
Batched: true
Location: InMemoryFileIndex [file:/workspace/data/cache/yellow_tripdata_2024-01.parquet, ... 1 entries]
PushedFilters: [IsNotNull(total_amount), GreaterThan(total_amount,50.0)]
ReadSchema: struct<VendorID:int,PULocationID:int,total_amount:double>

(2) ColumnarToRow [codegen id : 1]
Input [3]: [VendorID#256, PULocationID#263, total_amount#272]

(3) Filter [codegen id : 1]
Input [3]: [VendorID#256, PULocationID#263, total_amount#272]
Condition : (isnotnull(total_amount#272) AND (total_amount#272 > 50.0))

(4) Project [codegen id : 1]
Output [3]: [VendorID#256, total_amount#272, PULocationID#263]
Input [3]: [VendorID#256, PULocationID#263, total_amount#272]


== Physical Plan ==
* Project (4)
+- * Filter (3)
   +- * ColumnarToRow (2)
      +- Scan parquet  (1)


(1) Scan parquet 
Output [3]: 

**Column-pruning**: look at the *Scan parquet* operator; it only scans the necessary columns (`VendorID`, `PULocationID`, `total_amount`). 

## Example 3 — join then aggregate

In [16]:
from pyspark.sql import functions as F
z = zones.withColumnRenamed('LocationID', 'PULocationID')
joined_api = trips.join(z, 'PULocationID', 'left')
by_borough = joined_api.groupBy('Borough').agg(F.count('*').alias('n'))
by_borough.show(5)
explain_plan(by_borough)

+-------------+------+
|      Borough|     n|
+-------------+------+
|       Queens|523438|
|          EWR|   582|
|      Unknown| 20021|
|     Brooklyn| 53280|
|Staten Island|   129|
+-------------+------+
only showing top 5 rows
== Physical Plan ==
AdaptiveSparkPlan (11)
+- HashAggregate (10)
   +- Exchange (9)
      +- HashAggregate (8)
         +- Project (7)
            +- BroadcastHashJoin LeftOuter BuildRight (6)
               :- Scan parquet  (1)
               +- BroadcastExchange (5)
                  +- Project (4)
                     +- Filter (3)
                        +- Scan csv  (2)


(1) Scan parquet 
Output [1]: [PULocationID#263]
Batched: true
Location: InMemoryFileIndex [file:/workspace/data/cache/yellow_tripdata_2024-01.parquet, ... 1 entries]
ReadSchema: struct<PULocationID:int>

(2) Scan csv 
Output [2]: [LocationID#292, Borough#293]
Batched: false
Location: InMemoryFileIndex [file:/workspace/data/cache/taxi_zone_lookup.csv]
PushedFilters: [IsNotNull(LocationI

## Spark UI: jobs, stages, tasks

Run `by_borough.show()` — note the **job** in the UI. Transformations alone only extend the plan.

In [17]:
spark.conf.set('spark.sql.shuffle.partitions', '8')
by_borough.explain('formatted')
spark.conf.set('spark.sql.shuffle.partitions', '200')
by_borough.explain('formatted')

== Physical Plan ==
AdaptiveSparkPlan (11)
+- HashAggregate (10)
   +- Exchange (9)
      +- HashAggregate (8)
         +- Project (7)
            +- BroadcastHashJoin LeftOuter BuildRight (6)
               :- Scan parquet  (1)
               +- BroadcastExchange (5)
                  +- Project (4)
                     +- Filter (3)
                        +- Scan csv  (2)


(1) Scan parquet 
Output [1]: [PULocationID#263]
Batched: true
Location: InMemoryFileIndex [file:/workspace/data/cache/yellow_tripdata_2024-01.parquet, ... 1 entries]
ReadSchema: struct<PULocationID:int>

(2) Scan csv 
Output [2]: [LocationID#292, Borough#293]
Batched: false
Location: InMemoryFileIndex [file:/workspace/data/cache/taxi_zone_lookup.csv]
PushedFilters: [IsNotNull(LocationID)]
ReadSchema: struct<LocationID:string,Borough:string>

(3) Filter
Input [2]: [LocationID#292, Borough#293]
Condition : isnotnull(LocationID#292)

(4) Project
Output [2]: [LocationID#292 AS PULocationID#409, Borough#293]
Input [2

## Takeaways

1. DataFrame API and Spark SQL converge on the same planner.
2. Use `explain('formatted')` in class; `extended` shows analyzer/optimizer output.
3. Physical plan + UI connects code to cluster work (modules 02–05 go deeper).